# Sharing Jobs with dapi

Share an existing job (read-only) with collaborators or a whole DesignSafe project team, inspect the grants, and revoke them. Grantees can see the job record, history, inputs, and outputs.

**Requires dapi >= 0.5.5** (currently on the `dev` branch).

In [ ]:
%pip install --upgrade --quiet "git+https://github.com/DesignSafe-CI/dapi.git@dev"

In [ ]:
import dapi
from dapi import DSClient

print(
    f"dapi version: {dapi.__version__}"
)  # needs >= 0.5.5; restart kernel after upgrading

ds = DSClient()

### Pick a job to share

Any job you own works — no need to run a new one.

In [ ]:
df = ds.jobs.list(limit=5)
df[["name", "uuid", "status", "appId", "created_dt"]]

In [ ]:
# Use the most recent job, or paste a specific UUID
job = ds.jobs.job(df.iloc[0]["uuid"])
print(f"{job.uuid}: {job.status}")

### Share with a user

The username is validated against the tenant before any grant is issued — a typo raises an error and shares nothing. The default grant covers job history, inputs, outputs, and the resubmit request (READ only).

In [ ]:
job.share(user_id="parduino")

In [ ]:
# Current grants on this job
job.shares

### Share with a whole project team

Resolves every member (PI, co-PIs, team) of a DesignSafe project — preview with `ds.projects.members` first.

In [ ]:
# ds.projects.members("PRJ-XXXX")
# job.share(project_id="PRJ-XXXX")

### The grantee's side

Collaborators find jobs shared with them (run this as the *grantee*):

In [ ]:
shared = ds.jobs.list(list_type="SHARED_JOBS")
shared[["name", "uuid", "status", "appId", "created_dt"]] if len(shared) else shared

### Revoke

In [ ]:
job.unshare(user_id="parduino")
job.shares

**Note:** share grants work through the Tapis *jobs* service. dapi's output methods (`get_results`, `get_output_content`, ...) currently read through the *files* service, which does not see job shares — so grantees should view outputs of MyData-archived jobs in the DesignSafe portal (Workspace > Job Status) for now, or the owner should archive to a shared project. Routing dapi's output methods through the share-aware jobs endpoints is planned.